# Session 6 — DEIM and its error amplification

Download the notebook with the toolbar. Run the supplied baseline from a fresh kernel before changing settings.
Use [the course Python environment](https://feelpp.github.io/course-rom/course-rom/setup.html). Each practical starts independently of your earlier notebooks.
Read [the accompanying notes](https://feelpp.github.io/course-rom/rom/hyper-reduction/deim.html) for assumptions and derivations.
The timed tasks below occupy 60 minutes, including the closing comparison; optional extensions are outside that budget.
Website plots come from executing these same cells. Synthetic truth is used to evaluate methods, never as an undeclared estimator input.
## Snapshot POD (10 minutes)

The POD basis here approximates the source term, not a PDE state.
The S1 PCA connection applies to the snapshot matrix, but we keep these fields uncentered for a linear source space.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
n=120
x=np.linspace(0,1,n)
def source(points,center,width):
    return .1+np.exp(-((np.asarray(points)-center)/width)**2)
params=[(c,w) for c in np.linspace(.2,.8,9) for w in (.08,.14,.2)]
tests=[(.275,.11),(.425,.17),(.675,.13)]
S=np.column_stack([source(x,*p) for p in params])


## Select interpolation coordinates (20 minutes)

**Task 1.** Explain why the algorithm interpolates each new mode using previous modes before selecting its next point.
Test that all retained modes are reproduced.


In [ ]:
def deim(basis):
    indices=[]
    for k in range(basis.shape[1]):
        residual=basis[:,k].copy()
        if k: residual-=basis[:,:k]@np.linalg.solve(basis[indices,:k],basis[indices,k])
        i=int(np.argmax(np.abs(residual)))
        if abs(residual[i])<1e-12: raise ValueError('Dependent interpolation direction')
        indices.append(i)
    return np.array(indices)
U,s,_=np.linalg.svd(S,full_matrices=False)
m=8
Um=U[:,:m]
points=deim(Um)
T=Um[points,:]


## Projection error versus DEIM error (20 minutes)

**Task 2.** Verify the Euclidean bound with an absolute roundoff tolerance, then vary the retained rank.


In [ ]:
amplification=1/np.linalg.svd(T,compute_uv=False)[-1]
projection_errors=[]; deim_errors=[]
for p in tests:
    v=source(x,*p)
    projection_errors.append(np.linalg.norm(v-Um@(Um.T@v)))
    deim_errors.append(np.linalg.norm(v-Um@np.linalg.solve(T,v[points])))
print('DEIM inverse norm:',amplification)
print('DEIM bound check:',np.all(np.array(deim_errors)<=amplification*np.array(projection_errors)+1e-11))
fig,ax=plt.subplots(figsize=(7,3.5))
ax.semilogy(range(len(tests)),projection_errors,'o-',label='POD projection error')
ax.semilogy(range(len(tests)),deim_errors,'s-',label='DEIM error')
ax.semilogy(range(len(tests)),amplification*np.array(projection_errors),'--',label='DEIM bound')
ax.set(xlabel='Held-out case',ylabel='Euclidean source error'); ax.legend()
fig.tight_layout(); plt.show()


## Checkpoint (10 minutes)

Submit the bound comparison and selected coordinates.
**Task 3.** Explain why snapshot singular values alone cannot certify this held-out source error.
Optional: replace the greedy points by equally spaced ones and compare invertibility and amplification without claiming one rule always wins.
